In [ ]:
# Importing Libraries..
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

from torchvision.models import (
    resnet34,
    ResNet34_Weights
)

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
# Device Setup..

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
# Dataset..

mean = (0.4914, 0.4822, 0.4465)
std = (0.247, 0.243, 0.261)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_dataset = torchvision.datasets.CIFAR10(
    root=r'./data',
    train=False,
    download=False,
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print(len(test_dataset))

In [ ]:
# Gausaian blur..

class GaussianBlur(nn.Module):

    def __init__(self, channels):

        super(GaussianBlur, self).__init__()

        kernel = torch.tensor([
            [1.,2.,1.],
            [2.,4.,2.],
            [1.,2.,1.]
        ])

        kernel /= 16.0

        kernel = kernel.view(1,1,3,3)

        kernel = kernel.repeat(channels,1,1,1)

        self.weight = nn.Parameter(
            kernel,
            requires_grad=False
        )

        self.groups = channels

    def forward(self, x):

        return F.conv2d(
            x,
            self.weight,
            padding=1,
            groups=self.groups
        )

In [ ]:
# Denoise Blocck..

class DenoiseBlock(nn.Module):

    def __init__(self, channels):

        super(DenoiseBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):

        residual = x

        out = F.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        out = self.bn2(
            self.conv2(out)
        )

        out += residual

        out = F.relu(out)

        return out

In [ ]:
# AvgMaxPool..

class AvgMaxPool(nn.Module):

    def __init__(self):

        super(AvgMaxPool, self).__init__()

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.maxpool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):

        avg = self.avgpool(x)

        max_ = self.maxpool(x)

        return avg + max_

In [ ]:
# Model Building..

class SecureResNet34(nn.Module):

    def __init__(self, num_classes=10):

        super(SecureResNet34, self).__init__()

        self.backbone = resnet34(
            weights=ResNet34_Weights.DEFAULT
        )

        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()

        self.gaussian = GaussianBlur(64)

        self.denoise = DenoiseBlock(512)

        self.pool = AvgMaxPool()

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):

        x = self.backbone.conv1(x)

        x = self.backbone.bn1(x)

        x = self.backbone.relu(x)

        x = self.gaussian(x)

        x = self.backbone.layer1(x)

        x = self.backbone.layer2(x)

        x = self.backbone.layer3(x)

        x = self.backbone.layer4(x)

        x = self.denoise(x)

        x = self.pool(x)

        x = torch.flatten(x,1)

        x = self.fc(x)

        return x

In [ ]:
# loading Trainned model..

model = SecureResNet34().to(device)

model.load_state_dict(
    torch.load(
        "best_secure_resnet34_pretrained_pgd3.pth",
        map_location=device
    )
)

model.eval()

print("Model Loaded Successfully")

In [ ]:
# Clean Accuracy..
correct = 0

total = 0

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

clean_acc = 100. * correct / total

print(f"Clean Accuracy: {clean_acc:.2f}%")

In [ ]:
# FGSM ATTACK FUNCTION..

def fgsm_attack(model,
                images,
                labels,
                eps):

    images = images.clone().detach().to(device)

    labels = labels.to(device)

    images.requires_grad = True

    outputs = model(images)

    loss = nn.CrossEntropyLoss()(outputs, labels)

    model.zero_grad()

    loss.backward()

    adv_images = images + eps * images.grad.sign()

    adv_images = torch.clamp(
        adv_images,
        min=-1,
        max=1
    )

    return adv_images

In [ ]:
# FGSM EVALUATION...

fgsm_epsilons = [0.01, 0.03, 0.05, 0.07]

fgsm_results = []

for eps in fgsm_epsilons:

    correct = 0

    total = 0

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        adv_images = fgsm_attack(
            model,
            images,
            labels,
            eps
        )

        outputs = model(adv_images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total

    fgsm_results.append(acc)

    print(f"\nFGSM eps={eps}")

    print(f"Accuracy: {acc:.2f}%")

In [ ]:
# PGD Attck function..

def pgd_attack(model,
               images,
               labels,
               eps,
               alpha=2/255,
               steps=7):

    images = images.clone().detach().to(device)

    labels = labels.to(device)

    adv_images = images.clone().detach()

    adv_images = adv_images + torch.empty_like(
        adv_images
    ).uniform_(-eps, eps)

    adv_images = torch.clamp(
        adv_images,
        min=-1,
        max=1
    )

    for _ in range(steps):

        adv_images.requires_grad = True

        outputs = model(adv_images)

        loss = nn.CrossEntropyLoss()(outputs, labels)

        grad = torch.autograd.grad(
            loss,
            adv_images
        )[0]

        adv_images = adv_images.detach() + alpha * grad.sign()

        delta = torch.clamp(
            adv_images - images,
            min=-eps,
            max=eps
        )

        adv_images = torch.clamp(
            images + delta,
            min=-1,
            max=1
        ).detach()

    return adv_images

In [ ]:
pgd_epsilons = [0.01, 0.03, 0.05, 0.07]

In [ ]:
#PGD Evaluation..pgd_epsilons = [0.01, 0.03, 0.05, 0.07]

pgd_results = []

for eps in pgd_epsilons:

    correct = 0

    total = 0

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        adv_images = pgd_attack(
            model,
            images,
            labels,
            eps,
            steps=7
        )

        outputs = model(adv_images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total

    pgd_results.append(acc)

    print(f"\nPGD eps={eps}")

    print(f"Accuracy: {acc:.2f}%")

In [ ]:
# Deepfoll..

from art.attacks.evasion import DeepFool

from art.estimators.classification import PyTorchClassifier

In [ ]:
# Create Art classifier..

classifier = PyTorchClassifier(

    model=model,

    loss=nn.CrossEntropyLoss(),

    optimizer=None,

    input_shape=(3, 32, 32),

    nb_classes=10,

    clip_values=(-1, 1)

)

print("ART Classifier Ready")

In [ ]:
# Create Deepfool Attack..

deepfool_attack = DeepFool(

    classifier=classifier,

    max_iter=20,

    epsilon=1e-6

)

print("DeepFool Ready")

In [ ]:
# Deepfooll Evaluation..

correct = 0

total = 0

for images, labels in tqdm(test_loader):

    images_np = images.numpy()

    labels_np = labels.numpy()

    adv_images = deepfool_attack.generate(
        x=images_np
    )

    adv_images = torch.tensor(
        adv_images
    ).float().to(device)

    labels = torch.tensor(
        labels_np
    ).to(device)

    outputs = model(adv_images)

    _, predicted = outputs.max(1)

    total += labels.size(0)

    correct += predicted.eq(labels).sum().item()

deepfool_acc = 100. * correct / total

print(f"DeepFool Accuracy: {deepfool_acc:.2f}%")